In [1]:
import pandas as pd
import numpy as np

In [2]:
ENTREPOT_PATH = '/home/tbadie/Bureau/data/data_entrepot_outils/'
donnees = {}

def import_dfs(df_names, path_data, sep = ','):
    i = 0
    for df_name in df_names: 
        donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, low_memory=False).replace({'\r\n': '\n'}, regex=True)

tables = [
    'sdc',
    'entite_unique_par_sdc_nettoyage',
    'synthetise',
    'connection_synthetise',
    'connection_synthetise_restructure',
    'noeuds_synthetise',
    'noeuds_synthetise_restructure',
    'connection_realise',
    'noeuds_realise',
    'zone',
    'parcelle',
    'composant_culture',
    'espece',

    'poids_connexions_synthetise_rotation',
    'poids_noeuds_realise_outils_dirodur',
    'date_de_semis_outils_dirodur'
    ]

# import des données
import_dfs(tables, ENTREPOT_PATH, sep = ',')

In [ ]:
poids_S = donnees['poids_connexions_synthetise_rotation'][['connexion_id','poids_conx_agregation_norm_synth']].rename(columns={'connexion_id':'connection_synthetise_id'}).copy()
poids_R = donnees['poids_noeuds_realise'][['noeuds_realise_id','poids_surface_developpee_normalisee']].copy()
date_semis = donnees['date_de_semis_outils_dirodur'][['culture_id','saison_semis_detect_via_intv']].copy()

sdc = donnees['sdc'][['id','filiere']].rename(columns={'id':'sdc_id'}).copy()

unique_sdc = donnees['entite_unique_par_sdc_nettoyage'].copy()
sdc_real = sdc.loc[sdc['sdc_id'].isin(unique_sdc.loc[unique_sdc['entite_retenue'] == 'realise_retenu','sdc_id'])]
synthetise = donnees['synthetise'][['id','sdc_id']].rename(columns={'id':'synthetise_id'}).copy()
synthetise = synthetise.loc[synthetise['synthetise_id'].isin(unique_sdc['entite_retenue'].unique())]

cnx_s = donnees['connection_synthetise'][['id','cible_noeuds_synthetise_id']].rename(columns={'id':'connection_synthetise_id', 'cible_noeuds_synthetise_id':'noeuds_synthetise_id'}).copy()
cnx_s_rest = donnees['connection_synthetise_restructure'].rename(columns={'id':'connection_synthetise_id'}).copy()
nd_s = donnees['noeuds_synthetise'][['id','synthetise_id']].rename(columns={'id':'noeuds_synthetise_id'}).copy()
nd_s_rest = donnees['noeuds_synthetise_restructure'].rename(columns={'id':'noeuds_synthetise_id'}).copy()

cnx_r = donnees['connection_realise'][['id','cible_noeuds_realise_id','culture_intermediaire_id']].rename(columns={'id':'connexion_realise_id','cible_noeuds_realise_id':'noeuds_realise_id'})
nd_r = donnees['noeuds_realise'].rename(columns={'id':'noeuds_realise_id'}).copy()
zone = donnees['zone'][['id','parcelle_id']].rename(columns={'id':'zone_id'}).copy()
parcelle = donnees['parcelle'][['id','sdc_id']].rename(columns={'id':'parcelle_id'}).copy()

cropsp = donnees['composant_culture'][['id','espece_id','culture_id']].rename(columns={'id':'composant_culture_id'}).copy()
sp = donnees['espece'][['id','libelle_espece_botanique','typodirodur_espece','typodirodur_espece_precise','typodirodur_espece_famille_bota','typodirodur_espece_periode_semis']].rename(columns={'id':'espece_id'}).copy()
sp = cropsp.merge(sp, how = 'left', on = 'espece_id')

sp['ponderation_composant'] = 1/sp.groupby('culture_id')['composant_culture_id'].transform("count")

# merge outer pour les noeud sur les connexion en réalisé car tous les noeuds n'ont pas forcément de connexion
# merge inner avec synthetise et sdc pour n'avoir que les entite unique par sdc !
itk_s = cnx_s.merge(cnx_s_rest, how='left', on='connection_synthetise_id').merge(nd_s, how='left', on='noeuds_synthetise_id').merge(nd_s_rest, how='left', on='noeuds_synthetise_id').merge(synthetise, how='inner', on='synthetise_id')
itk_r = cnx_r.merge(nd_r, how='outer', on ='noeuds_realise_id').merge(zone, how='left', on='zone_id').merge(parcelle, how='left', on='parcelle_id').merge(sdc_real, how='inner', on='sdc_id')
itk = pd.concat([itk_s, itk_r])

itk = itk[['connection_synthetise_id', 'noeuds_realise_id', 'culture_id', 'culture_intermediaire_id', 'synthetise_id', 'sdc_id']]

In [4]:
composant_itk = itk.merge(sp, on='culture_id', how='left')
composant_itk = composant_itk.merge(poids_S, how='left', on='connection_synthetise_id')
composant_itk = composant_itk.merge(poids_R, how='left', on='noeuds_realise_id')

composant_itk = composant_itk.merge(date_semis, how='left', on='culture_id')
composant_itk['saison_semis_detect_via_intv'] = composant_itk['typodirodur_espece_periode_semis'].fillna(composant_itk['saison_semis_detect_via_intv'])
composant_itk.drop(columns = 'saison_semis_detect_via_intv', inplace=True)

In [ ]:
# Normalement pas besoin (checké en ipynb) mais c'est pour etre sûr que le poids n'est que d'une méthode ou d'une autre (S ou R)
composant_itk.loc[composant_itk['connection_synthetise_id'].isna(), 'poids_conx_agregation_norm_synth'] = pd.NA
composant_itk.loc[composant_itk['noeuds_realise_id'].isna(), 'poids_surface_developpee_normalisee'] = pd.NA